# Data Restructuring & Saving with Inferred Dates

In this notebook, we will:
- Modify the **Spike Path DataFrame** using **inferred dates** obtained from running Chronumental on the phylogenetic tree.
- Add unique **hashes** for efficient storage and quick lookup of:
  - Sequence paths
  - Mutation paths

This approach leverages more accurate, model-inferred dates rather than raw metadata dates, enabling:
- Improved temporal alignment with evolutionary events.
- More precise downstream analyses and storage

In [1]:
# --- VaccineScout configuration -------------------------------------------
# Makes scripts/ importable regardless of where the kernel was started, then
# loads every path and parameter from config.yaml at the repository root.
import sys
from pathlib import Path

_SCRIPTS_DIR = Path.cwd() if (Path.cwd() / "config_loader.py").is_file() else Path.cwd() / "scripts"
_SCRIPTS_DIR = _SCRIPTS_DIR.resolve()
if str(_SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(_SCRIPTS_DIR))

from config_loader import load_config

cfg = load_config()
print(f"Loaded config from {cfg.config_path}")

# Data handling
import pandas as pd
from datetime import datetime
import pickle
from tqdm import tqdm

# Sequence parsing
from Bio import SeqIO

# --- Custom imports ---
from Helpers import *   # Contains utility functions for data processing


Loaded config from /home/pgangwar@AD.UCSD.EDU/FutureCoV/config.yaml


In [2]:
# Load reference spike sequence
ROOT_PATH = cfg.data_root_str
refSeq = getReference(cfg.file("ref_spike_genome"))

# --- Load inferred dates and spike path data ---
dates_df = pd.read_csv(cfg.file("chronumental_dates"), sep='\t')

spike_paths_df = pd.read_parquet(cfg.file("node_spike_cumulative"))

print(f"Loaded dates_df with {dates_df.shape[0]} rows and {dates_df.shape[1]} columns")
print(f"Loaded spike_paths_df with {spike_paths_df.shape[0]} rows and {spike_paths_df.shape[1]} columns")

Loaded dates_df with 21096844 rows and 2 columns
Loaded spike_paths_df with 21096844 rows and 2 columns


In [3]:
# --- Drop internal nodes ---
initial_rows = spike_paths_df.shape[0]
spike_paths_df = spike_paths_df[~spike_paths_df['sample_id'].str.contains('node_')]
dropped_rows = initial_rows - spike_paths_df.shape[0]
print(f"Dropped {dropped_rows} rows containing internal nodes in node_id")
print(f"Remaining spike_paths_df has {spike_paths_df.shape[0]} rows")

Dropped 3961557 rows containing internal nodes in node_id
Remaining spike_paths_df has 17135287 rows


In [4]:
# --- Preprocess dates_df ---

# Convert 'predicted_date' column to datetime (ensure uniform format)
dates_df['predicted_date'] = pd.to_datetime(dates_df['predicted_date'])

# Extract date part only (discard time information)
dates_df['predicted_date'] = dates_df['predicted_date'].dt.date
dates_df['predicted_date'] = pd.to_datetime(dates_df['predicted_date'])

# Rename 'strain' column to 'node_id' for clarity
dates_df = dates_df.rename(columns={'strain': 'node_id'})

# Quick sanity check
print(f"Dates dataframe columns: {dates_df.columns.tolist()}")
print(f"Sample dates:")
print(dates_df.head())

Dates dataframe columns: ['node_id', 'predicted_date']
Sample dates:
                                          node_id predicted_date
0                                          node_1     2019-12-07
1      CHN/YN-0306-466/2020|MT396241.1|2020-03-06     2020-03-05
2                    DP0803|LC571037.1|2020-02-17     2020-02-16
3                                          node_2     2020-01-24
4  England/LEED-2A8B52/2020|OA971832.1|2020-04-04     2020-04-04


In [5]:
# --- Merge inferred dates into spike_paths_df ---

# Ensure matching key types as strings
dates_df['node_id'] = dates_df['node_id'].astype(str)
spike_paths_df['sample_id'] = spike_paths_df['sample_id'].astype(str)

# Merge inferred dates on 'sample_id'
spike_paths_df = spike_paths_df.merge(
    dates_df.rename(columns={'node_id': 'sample_id', 'predicted_date': 'date'}),
    on='sample_id',
    how='left'
)
print(f"Remaining spike_paths_df has {spike_paths_df.shape[0]} rows")

Remaining spike_paths_df has 17135287 rows


In [6]:
# --- Finalize date column and filter spike_paths_df by date range ---

# Ensure 'date' column is datetime (coerce errors to NaT)
spike_paths_df['date'] = pd.to_datetime(spike_paths_df['date'], errors='coerce')

# Sort by date ascending
spike_paths_df = spike_paths_df.sort_values(by='date', ascending=True).reset_index(drop=True)

# Define date range boundaries
start_date = pd.Timestamp(cfg.start_date)
end_date = pd.Timestamp(cfg.end_date)

# Filter to retain rows within the specified date range
spike_paths_df = spike_paths_df[
    (spike_paths_df['date'] >= start_date) & (spike_paths_df['date'] <= end_date)
].reset_index(drop=True)

print(f"Filtered spike_paths_df to {len(spike_paths_df)} entries between {start_date.date()} and {end_date.date()}")

Filtered spike_paths_df to 17134695 entries between 2020-01-01 and 2025-09-01


# Morphing Data: Creating Hashed Mutation and Sequence IDs

In this step, we will generate unique hash identifiers for:

- **Mutation sets**: to create compact Mutation IDs representing each unique mutation combination.
- **Sequences**: to create Sequence IDs that enable efficient storage and quick lookups.

This hashing significantly reduces data size and improves performance of downstream operations such as:

- Data merging and grouping.
- Fast mutation and sequence comparisons.
- Storage and retrieval optimizations.

The hash IDs serve as stable keys that abstract away verbose mutation or sequence details.

In [7]:
from tqdm import tqdm
import pandas as pd

tqdm.pandas()  # enable progress_apply / progress_map

def flatten_nested_array(val):
    """
    Converts standard mutation strings into a list format.
    """
    # 1. Check for missing data (NaN) or empty strings/spaces
    if pd.isna(val) or str(val).strip() == "":
        return []
    
    # 2. Return the string wrapped in a list. 
    # (Using .split(';') safely handles rows that might have multiple mutations like "S:P812L;S:A831V")
    return [m.strip() for m in str(val).split(';')]

# Apply with progress bar
spike_paths_df['aa_mutations'] = spike_paths_df['aa_mutations'].progress_map(flatten_nested_array)

100%|██████████████████████████████████████████████████████████| 17134695/17134695 [01:40<00:00, 171036.38it/s]


In [8]:
from tqdm import tqdm
tqdm.pandas(desc="Applying path mutations")

def apply_path_mutations(path, refSeq):
    """
    Apply a list of mutations (path) to the reference sequence.

    Args:
        path (list): List of mutation strings.
        refSeq (str or Seq): Reference sequence.
    
    Returns:
        str: Mutated sequence string.
    """
    if not path:  # empty or None path
        return str(refSeq)
    return apply_mutations_flat_list(path, refSeq)

# Cache dictionary to avoid repeated computations
cache = {}

def mutate_row_cached(path):
    """
    Mutate a sequence for a given mutation path using caching for speed.

    Args:
        path (list): List of mutation strings representing mutation path.

    Returns:
        str: Mutated sequence string.
    """
    key = tuple(path)  # immutable key for caching
    if key not in cache:
        cache[key] = apply_path_mutations(path, refSeq)
    return cache[key]

# Apply mutation paths with progress bar, store results in new column
spike_paths_df['mutated_sequence'] = spike_paths_df['aa_mutations'].progress_apply(mutate_row_cached)

Applying path mutations: 100%|█████████████████████████████████| 17134695/17134695 [01:02<00:00, 276035.13it/s]


# Generating Consistent Mutation and Sequence Keys

To facilitate fast and reliable lookups, we create unique keys for each:

- **Mutation set**:  
  Each unique combination of mutations is converted into a consistent string, hashed, and prefixed with `M_` to form a compact Mutation Key.

- **Sequence**:  
  Each mutated amino acid sequence is hashed and prefixed with `S_` to generate a unique Sequence Key.

These keys provide:  
- Fixed-length identifiers for easy indexing and storage.  
- Consistency across datasets to avoid redundancy.  
- Improved efficiency for join and comparison operations.

In [9]:
# --- Generate Mutation_ID and Sequence_ID columns ---

# Convert parsed spike mutation lists to tuples for hashing and uniqueness
spike_paths_df['mutation_path_tuple'] = spike_paths_df['aa_mutations'].apply(tuple)

# Create a unique list of mutation paths and assign Mutation_IDs
unique_paths = spike_paths_df['mutation_path_tuple'].unique()
mutation_id_map = {path: f"M_{i}" for i, path in enumerate(unique_paths)}
spike_paths_df['Mutation_ID'] = spike_paths_df['mutation_path_tuple'].apply(lambda x: mutation_id_map[x])

# Create a unique list of mutated sequences and assign Sequence_IDs
unique_seqs = spike_paths_df['mutated_sequence'].unique()
sequence_id_map = {seq: f"S_{i}" for i, seq in enumerate(unique_seqs)}
spike_paths_df['Sequence_ID'] = spike_paths_df['mutated_sequence'].apply(lambda x: sequence_id_map[x])

IOStream.flush timed out


In [10]:
# --- Create final_df and reverse lookup maps ---
final_df = spike_paths_df[['sample_id', 'Mutation_ID', 'Sequence_ID', 'date']].copy()
final_df.rename(columns={'sample_id': 'Sample_ID', 'date': 'Date'}, inplace=True)

# Reverse lookup dictionaries for sequences and mutation paths
sequence_dict = {v: k for k, v in sequence_id_map.items()}
mutation_path_dict = {v: k for k, v in mutation_id_map.items()}

In [11]:
import pandas as pd
from pathlib import Path

# --- Define output directory and paths ---
cfg.data_root.mkdir(parents=True, exist_ok=True)  # Ensure directory exists

final_df_path = cfg.file("final_df")
sequence_dict_path = cfg.file("sequence_dict")
mutation_path_dict_path = cfg.file("mutation_path_dict")

# --- Save final_df DataFrame as Parquet ---
final_df.to_parquet(final_df_path, compression="snappy", index=False)

# --- Save dictionaries as Parquet (Requires conversion to DataFrame first) ---

# Convert sequence_dict: creates a table with 'key' and 'value' columns
pd.DataFrame(
    list(sequence_dict.items()), 
    columns=['key', 'value']
).to_parquet(sequence_dict_path, compression="snappy", index=False)

# Convert mutation_path_dict: creates a table with 'key' and 'value' columns
pd.DataFrame(
    list(mutation_path_dict.items()), 
    columns=['key', 'value']
).to_parquet(mutation_path_dict_path, compression="snappy", index=False)

print(f"Saved final_df to {final_df_path}")
print(f"Saved sequence_dict to {sequence_dict_path}")
print(f"Saved mutation_path_dict to {mutation_path_dict_path}")

Saved final_df to /home/pgangwar@AD.UCSD.EDU/FutureCoV/data_test/final_df.parquet
Saved sequence_dict to /home/pgangwar@AD.UCSD.EDU/FutureCoV/data_test/sequence_dict.parquet
Saved mutation_path_dict to /home/pgangwar@AD.UCSD.EDU/FutureCoV/data_test/mutation_path_dict.parquet
